In [1]:
# ============================================================
# META vs FOXA — K-MEANS + NLP
# ============================================================

# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer

# 2. LOAD DATA
df = pd.read_csv("META_vs_FOXA_Stock_Comparison.csv")

print("Dataset shape:", df.shape)
display(df.head())

# 3. CLEAN DATA
df = df.drop_duplicates()

# Convert numeric columns
for col in df.columns:
    if df[col].dtype == "object":
        temp = pd.to_numeric(
            df[col].astype(str).str.replace(",", ""),
            errors="coerce"
        )
        if temp.notna().mean() > 0.8:
            df[col] = temp

# 4. NUMERICAL FEATURES
num_cols = df.select_dtypes(include=np.number).columns.tolist()

# Remove ID-like columns
features = [
    c for c in num_cols
    if not any(x in c.lower() for x in ["id", "year", "month", "day"])
]

df[features] = df[features].fillna(df[features].median())

print("\nFeatures used:")
print(features)

# 5. STANDARDIZE
scaler = StandardScaler()
X = scaler.fit_transform(df[features])

# 6. FIND BEST K
scores = {}

for k in range(2, min(6, len(df))):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    scores[k] = silhouette_score(X, labels)

best_k = max(scores, key=scores.get)

print("\nBest K:", best_k)
print("Silhouette Score:", scores[best_k])

# 7. K-MEANS
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X)

# 8. PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

df["PCA1"] = X_pca[:, 0]
df["PCA2"] = X_pca[:, 1]

# 9. PLOT CLUSTERS
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="PCA1",
    y="PCA2",
    hue="Cluster",
    palette="viridis",
    s=100
)

plt.title("META vs FOXA - K-Means Clustering")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(True)
plt.show()

# 10. CLUSTER SUMMARY
print("\nCluster Summary:")
display(
    df.groupby("Cluster")[features].mean()
)

# ============================================================
# NLP
# ============================================================

# 11. FIND TEXT COLUMNS
text_cols = df.select_dtypes(include="object").columns.tolist()

# Remove cluster if present
text_cols = [
    c for c in text_cols
    if c != "Cluster"
]

print("\nText columns:", text_cols)

# 12. COMBINE TEXT
if len(text_cols) > 0:

    df["Text"] = (
        df[text_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )

    # 13. CLEAN TEXT
    def clean_text(text):
        text = text.lower()
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    df["Clean_Text"] = df["Text"].apply(clean_text)

    # 14. TF-IDF
    tfidf = TfidfVectorizer(
        stop_words="english",
        max_features=30
    )

    X_text = tfidf.fit_transform(
        df["Clean_Text"]
    )

    words = tfidf.get_feature_names_out()

    tfidf_scores = X_text.mean(axis=0).A1

    word_scores = pd.DataFrame({
        "Word": words,
        "TF_IDF": tfidf_scores
    }).sort_values(
        "TF_IDF",
        ascending=False
    )

    print("\nTop NLP Terms:")
    display(word_scores.head(20))

    # 15. NLP PLOT
    plt.figure(figsize=(10, 6))

    sns.barplot(
        data=word_scores.head(15),
        x="TF_IDF",
        y="Word"
    )

    plt.title("Top NLP Terms")
    plt.show()

else:

    print(
        "\nNo text columns found. "
        "NLP/TF-IDF requires text such as news headlines or descriptions."
    )

# 16. SAVE RESULTS
df.to_csv(
    "META_vs_FOXA_KMeans_NLP_Results.csv",
    index=False
)

print(
    "\nCompleted! Results saved as "
    "'META_vs_FOXA_KMeans_NLP_Results.csv'"
)

FileNotFoundError: [Errno 2] No such file or directory: 'META_vs_FOXA_Stock_Comparison.csv'